<a href="https://colab.research.google.com/github/Churdlez/BUS118s-ML_Basics/blob/main/customer_churn_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Data source: Synthetic educational customer dataset generated for this assignment.
# The dataset contains 1,200 realistic customer records.

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# Load the customer churn dataset
DATA_FILE = "customer_churn.csv"
df = pd.read_csv(DATA_FILE)

# Numerical and categorical features
numeric_features = [
    "age",
    "tenure_months",
    "monthly_usage_hours",
    "purchase_amount",
    "customer_service_calls",
    "late_payments"
]

categorical_features = [
    "region",
    "contract_type",
    "autopay"
]

features = numeric_features + categorical_features

X = df[features]
y = df["churn"]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Scale numerical features and encode categorical features
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

# Create the logistic regression pipeline
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

# Train the model
model.fit(X_train, y_train)

# Make predictions on the test data
test_predictions = model.predict(X_test)
test_probabilities = model.predict_proba(X_test)[:, 1]

# Evaluate the model
print(f"Number of records: {len(df)}")
print(f"Test accuracy: {accuracy_score(y_test, test_predictions):.3f}")
print(f"Test ROC-AUC: {roc_auc_score(y_test, test_probabilities):.3f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        test_predictions,
        zero_division=0
    )
)

# Create a new customer for prediction
new_customer = pd.DataFrame({
    "age": [35],
    "tenure_months": [8],
    "monthly_usage_hours": [20],
    "purchase_amount": [150],
    "customer_service_calls": [5],
    "late_payments": [2],
    "region": ["West"],
    "contract_type": ["Month-to-month"],
    "autopay": ["No"]
})

# Predict the probability of churn
churn_probability = model.predict_proba(new_customer)[0, 1]

# Classify the customer using a 0.50 threshold
threshold = 0.50
churn_prediction = int(churn_probability >= threshold)

print(
    f"\nChurn probability for new customer: "
    f"{churn_probability:.2%}"
)

print(
    f"Churn prediction using a {threshold:.2f} threshold "
    f"(1 = churn, 0 = no churn): {churn_prediction}"
)

# Display model coefficients
categorical_names = (
    model.named_steps["preprocessor"]
    .named_transformers_["categorical"]
    .get_feature_names_out(categorical_features)
)

feature_names = numeric_features + list(categorical_names)

coefficients = (
    model.named_steps["classifier"]
    .coef_[0]
)

print("\nModel Coefficients:")

for feature, coefficient in zip(feature_names, coefficients):
    direction = "increases" if coefficient > 0 else "decreases"

    print(
        f"{feature}: {coefficient:.3f} "
        f"({direction} estimated churn likelihood)"
    )

print("\nInterpretation:")
print(
    "The churn probability represents the estimated chance that "
    "a customer will leave the company."
)

print(
    "Customers with a probability of at least 50% are classified "
    "as likely to churn."
)

print(
    "Businesses can use these predictions to provide retention "
    "offers, customer support, or contract incentives."
)

Number of records: 1200
Test accuracy: 0.742
Test ROC-AUC: 0.762

Classification Report:
              precision    recall  f1-score   support

           0       0.64      0.38      0.47        74
           1       0.77      0.90      0.83       166

    accuracy                           0.74       240
   macro avg       0.70      0.64      0.65       240
weighted avg       0.73      0.74      0.72       240


Churn probability for new customer: 99.37%
Churn prediction using a 0.50 threshold (1 = churn, 0 = no churn): 1

Model Coefficients:
age: 0.144 (increases estimated churn likelihood)
tenure_months: -0.934 (decreases estimated churn likelihood)
monthly_usage_hours: -0.330 (decreases estimated churn likelihood)
purchase_amount: -0.266 (decreases estimated churn likelihood)
customer_service_calls: 0.558 (increases estimated churn likelihood)
late_payments: 0.514 (increases estimated churn likelihood)
region_East: -0.220 (decreases estimated churn likelihood)
region_North: -0.089 